In [479]:
import requests
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.keys import Keys #to be able to click/input
from selenium.webdriver.support.ui import WebDriverWait #wait excplicit
from selenium.webdriver.support import expected_conditions as EC #wait explicit
from selenium.webdriver.common.by import By #find_element
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException, ElementClickInterceptedException
from tabulate import tabulate
import pandas as pd
from pandas.errors import InvalidIndexError
import numpy as np

# *PROFIL DPR SIMPLIFIED*

In [ ]:
#DPR

base_url = "https://sirekap-obj-data.kpu.go.id/pemilu/caleg/partai/{:02d}{:02d}.json"
headers = {"^sec-ch-ua": "^\^Not"}

for i in range(11,93):
    for j in range(1, 10):
        url = base_url.format(i, j)
        response = requests.request("GET", url, headers=headers)
        if response.status_code != 404:
            print(response.text)

# *PROFIL DPR COMPLETE*

> ### *Preparation*

In [3]:
driver = webdriver.Chrome()
url = 'https://infopemilu.kpu.go.id/Pemilu/Dct_dpr'
driver.get(url)

> ### *Frameowork 1: Region Looping*

In [ ]:
# COMMAND HERE (works per 15/08/2024) #

DPR = pd.DataFrame() # for the DataFrame

for i in range(1, 85):
    driver.refresh();
    time.sleep(1)
    driver.find_element(By.XPATH, '//*[@id="register"]/div/span/span[1]/span').click()
    region = driver.find_elements(By.CSS_SELECTOR, 'li.select2-results__option')
    try:
        region[i].click()
    except IndexError as wait_region:
        driver.refresh();
        time.sleep(2)
        region[i].click()
    wait = WebDriverWait(driver, 20).until(EC.element_to_be_clickable((By.XPATH, '//*[@id="tbl_ms_nasional"]/tbody/tr[1]/td[9]/form/input[7]')))

    # loading profiles in each region
    profile = driver.find_elements(By.CSS_SELECTOR, 'tr.odd, tr.even')
    for x in profile:
        try:
            ###### Profile Scraping ######
        except NoSuchElementException as private_profile: 
            continue

**To review**
<br/>
Scrolldown is necessary and not yet implemented 
<br/><br/>
*Some Example*
<br/>
driver.execute_script("arguments[0].scrollIntoView();", x.find_element(By.CLASS_NAME, 'btn-secondary'))

> ### *Framework 2: Profile Scraping*

In [ ]:
#DPR = pd.DataFrame()

################################################################ Non-Table 
def get_element_text(xpath, blank=""):
    try:
        element = driver.find_element(By.XPATH, xpath)
        return element.text
    except NoSuchElementException as missing_non_table:
        return blank

contact = []

# General Identity
identity_xpath = "/html/body/div[2]/div[3]/div/div/div[1]/div/div[1]/div/div[4]/table"
identity_rows = get_element_text(identity_xpath).split('\n')
for row in identity_rows:
    contact.append(row)

# Address
address_xpath = '/html/body/div[2]/div[3]/div/div/div[1]/div/div[2]'
address_items = get_element_text(address_xpath).split('\n')
for item in address_items:
    contact.append(item)

status = []

# Job
job_xpath = '/html/body/div[2]/div[3]/div/div/div[1]/div/div[3]/p'
jobs = get_element_text(job_xpath)
status.append(jobs)

# Legal Status
legalstatus_xpath = '/html/body/div[2]/div[3]/div/div/div[1]/div/div[5]/p'
legal = get_element_text(legalstatus_xpath)
status.append(legal)

# Program Proposal
progprop_xpath = '/html/body/div[2]/div[3]/div/div/div[1]/div/div[11]/li/strong'
proposal = get_element_text(progprop_xpath)
status.append(proposal)

################################################################ Table 
table = []

def get_table_data(xpath, blank=""):
    try:
        table_element = driver.find_element(By.XPATH, xpath)
        rows = table_element.find_elements(By.XPATH, ".//tr")[1:]  
        return [row.text for row in rows]
    except NoSuchElementException as missing_table:
        return blank

# Previous Jobs
prev_job_data = get_table_data('/html/body/div[2]/div[3]/div/div/div[1]/div/div[4]/table')
table.append({'Riwayat Pekerjaan': prev_job_data})

# Previous Educations
prev_educ_data = get_table_data('/html/body/div[2]/div[3]/div/div/div[1]/div/div[6]/table')
table.append({'Riwayat Edukasi': prev_educ_data})

# Previous Courses
prev_course_data = get_table_data('/html/body/div[2]/div[3]/div/div/div[1]/div/div[7]/table')
table.append({'Riwayat Kursus & Diklat': prev_course_data})

# Previous Organizations
prev_org_data = get_table_data('/html/body/div[2]/div[3]/div/div/div[1]/div/div[8]/table')
table.append({'Riwayat Organisasi': prev_org_data})

# Previous Awards
prev_awards_data = get_table_data('/html/body/div[2]/div[3]/div/div/div[1]/div/div[9]/table')
table.append({'Riwayat Penghargaan': prev_awards_data})

# Combine tables into a single dictionary
dpr_table = {}
for entry in table:
    dpr_table.update(entry)

################################################################ DataFrame 
# Contact
contact_list = {x.split(': ')[0]: x.split(': ')[1] for x in contact if ': ' in x}
DPR_contact = pd.DataFrame([contact_list])

# Status
DPR_status = pd.DataFrame(columns=['Pekerjaan', 'Status Hukum', 'Program Usulan'])
length = len(DPR_status)
DPR_status.loc[length] = status

# Contact+Status
DPR_contus = pd.concat([DPR_contact, DPR_status], axis=1)

# Table
DPR_table = pd.DataFrame([dpr_table])

# Complete (Contact+Status+Table)
DPR_full = pd.concat([DPR_contus, DPR_table], axis=1)

# Appending the loops
DPR = pd.concat([DPR, DPR_full], ignore_index=True)

In [ ]:
# PATCH 1.A (on going) #

# DPR = pd.DataFrame()

card = driver.find_elements(By.CSS_SELECTOR, 'div.card')
for cards in card:
    non_table = cards.find_elements(By.CSS_SELECTOR, 'div.container:not(.table-responsive)')
    if non_table == []:
        del non_table
        continue
    else:
        break

for cards in card:
    table = cards.find_elements(By.CSS_SELECTOR, 'div.container.table-responsive')
    if table == []:
        del table
        continue
    else:
        break


############################## Non-Table: df_profile

## For all: df_profile
profiles_list = {}
for non in non_table:
    profil = non.find_elements(By.CSS_SELECTOR, 'table.table-striped > tbody > tr > td, li.list-group-item, h3.mt-3') + non.find_elements(By.TAG_NAME, 'p')
    profile = [prof.text.strip() for prof in profil]
    if 'ALAMAT' in profile:
        alamat = {x.split(': ')[0]: x.split(': ')[1] for x in profile if ': ' in x}
    elif 'PROGRAM USULAN' in profile:
        df_programs = pd.DataFrame({profile[0]: [profile[1:]]})
    else:
        for i in range(0, len(profile), 2):
            column_prof = profile[i]
            value_prof = profile[i+1]
            profiles_list[column_prof] = value_prof
        
df_profile_prof = pd.DataFrame([profiles_list])
df_profile_alamat = pd.DataFrame([alamat])
df_profile = pd.concat([df_profile_prof, df_profile_alamat, df_programs], axis=1)

    
############################## Table: df_table

## Table Headers: df_mini_header
dataframes = {}
for i, iden in enumerate(table, start=1):
    mini_header = iden.find_elements(By.CSS_SELECTOR, 'th')
    mini_headers = [mini.text for mini in mini_header]
    df_mini_header = pd.DataFrame(columns = mini_headers) # For each table, create a dataframe. Since some will be empty (for god knows why)....
    
    dataframes[f'df_{i}'] = df_mini_header #... let's try to include only those that are not empty ...
    df_table = [] #... and we fill the non-empty dataframe here
    for j in range(1, len(dataframes) + 1): 
        if dataframes[f'df_{j}'].columns.size > 0:
            df_table.append(dataframes[f'df_{j}'])

## Filling the Columns
iterate = 0 # To note in which dataframe is the iteration in later on
for tab in table:
    value = tab.find_elements(By.CSS_SELECTOR, 'td')
    values = [val.text for val in value]
    if values:  
        # Get the current DataFrame to update
        df = df_table[iterate]
        
        # Split values into chunks based on the number of columns in the DataFrame
        num_columns = len(df.columns)
        
        # If there is literally empty column (not even '-'), fill with empty columns
        if len(values) % num_columns != 0:
            total_values = len(values)
            required_values = (total_values // num_columns + 1) * num_columns # this line is fucking smart. fucking ai man
            values += [''] * (required_values - total_values)

        # Split the rows to fit into the dataframe
        rows = [values[i:i + num_columns] for i in range(0, len(values), num_columns)]
            
        # Create a new DataFrame from the rows and append to the current DataFrame
        df_new = pd.DataFrame(rows, columns=df.columns)
        df_table[iterate] = pd.concat([df, df_new], ignore_index=True)
        
        # Move to the next DataFrame for the next set of values
        iterate += 1
    
    # Break the loop if all DataFrames are populated
    if iterate >= len(df_table):
        break

## Filling the Columns: jobs (inserted to df_table)
for occ in table:
    job_info = occ.find_elements(By.TAG_NAME, 'h3') + occ.find_elements(By.TAG_NAME, 'p')
    jobs_info = [jobs.text.strip() for jobs in job_info]
    jobs = pd.DataFrame({jobs_info[0]:[jobs_info[1]]})
    df_table.append(jobs)
    if len(jobs_info)<=2:
        break

## concatenate the df_table
for i in range(0, len(df_table) - 1):
    if len(df_table[i])>1:
        df_table[i] = pd.DataFrame({col: [', '.join(df_table[i][col])] for col in df_table[i].columns})
    else:
        continue

df_table = pd.concat(df_table, axis=1) 

############################## Final
DPR_profile = pd.concat([df_profile, df_table], axis=1)
try:
    DPR = pd.concat([DPR, DPR_profile], ignore_index=True)
except InvalidIndexError as formatting:
    DPR = pd.concat([DPR, DPR_profile], ignore_index=True)

> ### *Framework 3: COMPLETE CODE*

In [ ]:
# COMMAND HERE VER 1 (works per 15/08/2024) #

DPR = pd.DataFrame() 

for i in range(1, 85):
    driver.refresh();
    time.sleep(1)
    driver.find_element(By.XPATH, '//*[@id="register"]/div/span/span[1]/span').click()
    region = driver.find_elements(By.CSS_SELECTOR, 'li.select2-results__option')
    try:
        region[i].click()
    except IndexError as wait_region:
        driver.refresh();
        time.sleep(2)
        region[i].click()
    wait = WebDriverWait(driver, 20).until(EC.element_to_be_clickable((By.XPATH, '//*[@id="tbl_ms_nasional"]/tbody/tr[1]/td[9]/form/input[7]')))

    # loading profiles in each region
    profile = driver.find_elements(By.CSS_SELECTOR, 'tr.odd, tr.even')
    for x in profile:
        try:
            wait = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.XPATH, '//*[@id="tbl_ms_nasional"]/tbody')))
            try:
                x.find_element(By.CLASS_NAME, 'btn-secondary').click()
            except StaleElementReferenceException as wait_back:
                time.sleep(1)
                driver.back();
                x.find_element(By.CLASS_NAME, 'btn-secondary').click()
            except ElementClickInterceptedException as scroll:
                driver.execute_script("window.scrollBy(0, 500);")
                time.sleep(1)
                x.find_element(By.CLASS_NAME, 'btn-secondary').click()
                
            ################################################################ Profile Scraping ##################################################################
            ################################################################ Non-Table 
            def get_element_text(xpath, blank=""):
                try:
                    element = driver.find_element(By.XPATH, xpath)
                    return element.text
                except NoSuchElementException as missing_non_table:
                    return blank
            
            contact = []
            
            # General Identity
            identity_xpath = "/html/body/div[2]/div[3]/div/div/div[1]/div/div[1]/div/div[4]/table"
            identity_rows = get_element_text(identity_xpath).split('\n')
            for row in identity_rows:
                contact.append(row)
            
            # Address
            address_xpath = '/html/body/div[2]/div[3]/div/div/div[1]/div/div[2]'
            address_items = get_element_text(address_xpath).split('\n')
            for item in address_items:
                contact.append(item)
            
            status = []
            
            # Job
            job_xpath = '/html/body/div[2]/div[3]/div/div/div[1]/div/div[3]/p'
            jobs = get_element_text(job_xpath)
            status.append(jobs)
            
            # Legal Status
            legalstatus_xpath = '/html/body/div[2]/div[3]/div/div/div[1]/div/div[5]/p'
            legal = get_element_text(legalstatus_xpath)
            status.append(legal)
            
            # Program Proposal
            progprop_xpath = '/html/body/div[2]/div[3]/div/div/div[1]/div/div[11]/li/strong'
            proposal = get_element_text(progprop_xpath)
            status.append(proposal)
            
            ################################################################ Table 
            table = []
            
            def get_table_data(xpath, blank=""):
                try:
                    table_element = driver.find_element(By.XPATH, xpath)
                    rows = table_element.find_elements(By.XPATH, ".//tr")[1:]  
                    return [row.text for row in rows]
                except NoSuchElementException as missing_table:
                    return blank
            
            # Previous Jobs
            prev_job_data = get_table_data('/html/body/div[2]/div[3]/div/div/div[1]/div/div[4]/table')
            table.append({'Riwayat Pekerjaan': prev_job_data})
            
            # Previous Educations
            prev_educ_data = get_table_data('/html/body/div[2]/div[3]/div/div/div[1]/div/div[6]/table')
            table.append({'Riwayat Edukasi': prev_educ_data})
            
            # Previous Courses
            prev_course_data = get_table_data('/html/body/div[2]/div[3]/div/div/div[1]/div/div[7]/table')
            table.append({'Riwayat Kursus & Diklat': prev_course_data})
            
            # Previous Organizations
            prev_org_data = get_table_data('/html/body/div[2]/div[3]/div/div/div[1]/div/div[8]/table')
            table.append({'Riwayat Organisasi': prev_org_data})
            
            # Previous Awards
            prev_awards_data = get_table_data('/html/body/div[2]/div[3]/div/div/div[1]/div/div[9]/table')
            table.append({'Riwayat Penghargaan': prev_awards_data})
            
            # Combine tables into a single dictionary
            dpr_table = {}
            for entry in table:
                dpr_table.update(entry)
            
            ################################################################ DataFrame 
            # Contact
            contact_list = {x.split(': ')[0]: x.split(': ')[1] for x in contact if ': ' in x}
            DPR_contact = pd.DataFrame([contact_list])
            
            # Status
            DPR_status = pd.DataFrame(columns=['Pekerjaan', 'Status Hukum', 'Program Usulan'])
            length = len(DPR_status)
            DPR_status.loc[length] = status
            
            # Contact+Status
            DPR_contus = pd.concat([DPR_contact, DPR_status], axis=1)
            
            # Table
            DPR_table = pd.DataFrame([dpr_table])
            
            # Complete (Contact+Status+Table)
            DPR_full = pd.concat([DPR_contus, DPR_table], axis=1)
            
            # Appending the loops
            DPR = pd.concat([DPR, DPR_full], ignore_index=True)
            ################################################################ Profile Scraping ##################################################################
            
            # Back
            driver.back(); 
        except NoSuchElementException as private_profile: 
            continue

In [ ]:
# PATCH 1.A (on going) #

DPR = pd.DataFrame() 

for i in range(1, 85):
    driver.refresh();
    time.sleep(1)
    driver.find_element(By.XPATH, '//*[@id="register"]/div/span/span[1]/span').click()
    region = driver.find_elements(By.CSS_SELECTOR, 'li.select2-results__option')
    try:
        region[i].click()
    except IndexError as wait_region:
        driver.refresh();
        time.sleep(2)
        region[i].click()
    wait = WebDriverWait(driver, 20).until(EC.element_to_be_clickable((By.XPATH, '//*[@id="tbl_ms_nasional"]/tbody/tr[1]/td[9]/form/input[7]')))

    # loading profiles in each region
    profile = driver.find_elements(By.CSS_SELECTOR, 'tr.odd, tr.even')
    for x in profile:
        try:
            wait = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.XPATH, '//*[@id="tbl_ms_nasional"]/tbody')))
            try:
                x.find_element(By.CLASS_NAME, 'btn-secondary').click()
            except StaleElementReferenceException as wait_back:
                time.sleep(1)
                driver.back();
                x.find_element(By.CLASS_NAME, 'btn-secondary').click()
            except ElementClickInterceptedException as scroll:
                driver.execute_script("window.scrollBy(0, 500);")
                time.sleep(1)
                x.find_element(By.CLASS_NAME, 'btn-secondary').click()
                
            ################################################################ Profile Scraping ##################################################################
            card = driver.find_elements(By.CSS_SELECTOR, 'div.card')
            for cards in card:
                non_table = cards.find_elements(By.CSS_SELECTOR, 'div.container:not(.table-responsive)')
                if non_table == []:
                    del non_table
                    continue
                else:
                    break
            
            for cards in card:
                table = cards.find_elements(By.CSS_SELECTOR, 'div.container.table-responsive')
                if table == []:
                    del table
                    continue
                else:
                    break
            
            
            ############################## Non-Table: df_profile
            
            ## For all: df_profile
            profiles_list = {}
            for non in non_table:
                profil = non.find_elements(By.CSS_SELECTOR, 'table.table-striped > tbody > tr > td, li.list-group-item, h3.mt-3') + non.find_elements(By.TAG_NAME, 'p')
                profile = [prof.text.strip() for prof in profil]
                if 'ALAMAT' in profile:
                    alamat = {x.split(': ')[0]: x.split(': ')[1] for x in profile if ': ' in x}
                elif 'PROGRAM USULAN' in profile:
                    df_programs = pd.DataFrame({profile[0]: [profile[1:]]})
                else:
                    for i in range(0, len(profile), 2):
                        column_prof = profile[i]
                        value_prof = profile[i+1]
                        profiles_list[column_prof] = value_prof
                    
            df_profile_prof = pd.DataFrame([profiles_list])
            df_profile_alamat = pd.DataFrame([alamat])
            df_profile = pd.concat([df_profile_prof, df_profile_alamat, df_programs], axis=1)
            
                
            ############################## Table: df_table
            
            ## Table Headers: df_mini_header
            dataframes = {}
            for i, iden in enumerate(table, start=1):
                mini_header = iden.find_elements(By.CSS_SELECTOR, 'th')
                mini_headers = [mini.text for mini in mini_header]
                df_mini_header = pd.DataFrame(columns = mini_headers) # For each table, create a dataframe. Since some will be empty (for god knows why)....
                
                dataframes[f'df_{i}'] = df_mini_header #... let's try to include only those that are not empty ...
                df_table = [] #... and we fill the non-empty dataframe here
                for j in range(1, len(dataframes) + 1): 
                    if dataframes[f'df_{j}'].columns.size > 0:
                        df_table.append(dataframes[f'df_{j}'])
            
            ## Filling the Columns
            iterate = 0 # To note in which dataframe is the iteration in later on
            for tab in table:
                value = tab.find_elements(By.CSS_SELECTOR, 'td')
                values = [val.text for val in value]
                if values:  
                    # Get the current DataFrame to update
                    df = df_table[iterate]
                    
                    # Split values into chunks based on the number of columns in the DataFrame
                    num_columns = len(df.columns)
                    
                    # If there is literally empty column (not even '-'), fill with empty columns
                    if len(values) % num_columns != 0:
                        total_values = len(values)
                        required_values = (total_values // num_columns + 1) * num_columns # this line is fucking smart. fucking ai man
                        values += [''] * (required_values - total_values)
            
                    # Split the rows to fit into the dataframe
                    rows = [values[i:i + num_columns] for i in range(0, len(values), num_columns)]
                        
                    # Create a new DataFrame from the rows and append to the current DataFrame
                    df_new = pd.DataFrame(rows, columns=df.columns)
                    df_table[iterate] = pd.concat([df, df_new], ignore_index=True)
                    
                    # Move to the next DataFrame for the next set of values
                    iterate += 1
                
                # Break the loop if all DataFrames are populated
                if iterate >= len(df_table):
                    break
            
            ## Filling the Columns: jobs (inserted to df_table)
            for occ in table:
                job_info = occ.find_elements(By.TAG_NAME, 'h3') + occ.find_elements(By.TAG_NAME, 'p')
                jobs_info = [jobs.text.strip() for jobs in job_info]
                jobs = pd.DataFrame({jobs_info[0]:[jobs_info[1]]})
                df_table.append(jobs)
                if len(jobs_info)<=2:
                    break
            
            ## concatenate the df_table
            for i in range(0, len(df_table) - 1):
                if len(df_table[i])>1:
                    df_table[i] = pd.DataFrame({col: [', '.join(df_table[i][col])] for col in df_table[i].columns})
                else:
                    continue
            
            df_table = pd.concat(df_table, axis=1) 
            
            ############################## Final
            DPR_profile = pd.concat([df_profile, df_table], axis=1)
            DPR = pd.concat([DPR, DPR_profile], axis=1, join='outer')
            ################################################################ Profile Scraping ##################################################################
            
            # Back
            driver.back(); 
        except NoSuchElementException as private_profile: 
            continue

In [108]:
# Export #
DPR.to_excel('Scraping KPU 150824.xlsx', index=False)

Bugs for Ver 1:
1. In some profiles, nama lengkap, tempat, tanggal lahir is in the rightmost of the columns (due to the incomplete profile or different spellings/cases}. Additionally, those profiles also have several mismatched columns to row.

Bugs for Ver 2:
1. Cannot handle if there's naming differences, even if it's just 1 capital letter difference

Improvement for Ver 1.B:
1. Make it click more than 1 for faster scraping
